# Data Collection
## Oil Shocks & Capital Reallocation Toward Green Firms

**Data sources**
- LSEG (Refinitiv) — daily OHLCV, turnover, market capitalisation, NAV, and shares outstanding
- Yahoo Finance — VIX and OVX volatility indices
- Kenneth French's data library — daily Fama-French 5-factor returns

**Sample period:** 2021-01-01 → 2026-08-30

---


## Setup & Configuration

In [1]:
import warnings
warnings.filterwarnings("ignore")

import io
import time
import zipfile
import requests
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
import lseg.data as ld

pd.set_option("display.max_columns", 200)
pd.set_option("display.width",       200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

In [2]:
START_DATE = "2021-01-01"
END_DATE   = "2026-08-30"

# ── Folder layout ──────────────────────────────────────────────────────────────
BASE_DIR  = Path.cwd()
RAW_DIR   = BASE_DIR / "../data/raw"
CLEAN_DIR = BASE_DIR / "../data/clean"

for folder in [RAW_DIR, CLEAN_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# ── Input files ────────────────────────────────────────────────────────────────
GREEN_FIRM_FILE = BASE_DIR / "../data/raw/RenewableCompaniesTicker.csv"
BROWN_FIRM_FILE = BASE_DIR / "../data/raw/Oil&GasCompaniesTicker.csv"

# ── LSEG field mnemonics ───────────────────────────────────────────────────────
MARKET_FIELDS = [
    "OPEN_PRC",            # open price
    "HIGH_1",              # daily high
    "LOW_1",               # daily low
    "TRDPRC_1",            # close price
    "ACVOL_UNS",           # volume (unadjusted)
    "TRNOVR_UNS",          # turnover (unadjusted)
    "TR.CompanyMarketCap"  # market capitalisation
]

COMMODITY_FIELDS = [
    "OPEN_PRC",
    "HIGH_1",
    "LOW_1",
    "TRDPRC_1",
    "ACVOL_UNS",
    "TRNOVR_UNS",
]

NAV_FIELDS    = ["TR.FundTotalNetAssets"]
SHARES_FIELDS = ["TR.SharesOutstanding"]

print(f"Sample period : {START_DATE} → {END_DATE}")
print(f"Raw dir       : {RAW_DIR}")
print(f"Clean dir     : {CLEAN_DIR}")

Sample period : 2021-01-01 → 2026-08-30
Raw dir       : /Users/shirindehghannezhad/Documents/Thesis Project/notebooks/../data/raw
Clean dir     : /Users/shirindehghannezhad/Documents/Thesis Project/notebooks/../data/clean


## Helper Functions

Reusable building blocks for downloading, reshaping, and saving data defined once and called throughout. 

In [3]:
# ── LSEG request with retry ────────────────────────────────────────────────────
def lseg_get_history(universe, fields, start, end, interval="1D", retries=3, pause=2):
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            return ld.get_history(
                universe=universe, fields=fields,
                start=start, end=end, interval=interval
            )
        except Exception as err:
            last_err = err
            if attempt < retries:
                time.sleep(pause)
    raise last_err


def lseg_get_data(universe, fields, parameters):
    return ld.get_data(universe=universe, fields=fields, parameters=parameters)


# ── Wide-to-long reshape ───────────────────────────────────────────────────────
def wide_to_long(df):
    df = df.copy().reset_index()
    df.columns = [str(c).strip() for c in df.columns]

    if isinstance(df.columns, pd.MultiIndex):
        # Layout A: (RIC, FIELD) multi-index
        records = []
        rics    = df.columns.get_level_values(0).unique()
        for ric in rics:
            sub = df[[c for c in df.columns if c[0] == ric]].copy()
            sub.columns = [c[1] for c in sub.columns]
            sub["ric"]  = str(ric).upper()
            sub["date"] = df.iloc[:, 0]
            records.append(sub)
        return pd.concat(records, ignore_index=True)
    else:
        # Layout B: single-RIC flat frame
        for alias in ["date", "index", "timestamp", "Date"]:
            if alias in df.columns:
                df = df.rename(columns={alias: "date"})
                break
        df["ric"] = ""
        return df


# ── Field rename ───────────────────────────────────────────────────────────────
FIELD_MAP = {
    "open_prc":            "open",
    "high_1":              "high",
    "low_1":               "low",
    "trdprc_1":            "close",
    "acvol_uns":           "volume",
    "trnovr_uns":          "turnover",
    "tr.companymarketcap": "market_cap",
    "company market cap":  "market_cap",  
}

def rename_fields(df):
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df.rename(columns=FIELD_MAP)


# ── Type coercion ──────────────────────────────────────────────────────────────
def coerce_types(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
    for col in ["open","high","low","close","volume","turnover","market_cap"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df.dropna(subset=["date"]).reset_index(drop=True)


# ── Feature engineering ────────────────────────────────────────────────────────
def add_features(df, group_col="ric"):
    df = df.sort_values([group_col, "date"]).copy()
    df["log_return"]    = df.groupby(group_col)["close"].transform(lambda s: np.log(s).diff())
    df["simple_return"] = df.groupby(group_col)["close"].transform(lambda s: s.pct_change())
    df["traded_value"]  = df["close"] * df["volume"]
    # Parkinson (1980) intraday volatility estimator
    df["log_hl_range"]    = np.log(df["high"] / df["low"])
    df["parkinson_vol"]   = df["log_hl_range"] ** 2 / (4 * np.log(2))
    return df


# ── Save helper ────────────────────────────────────────────────────────────────
def save_csv(df, folder, filename):
    path = Path(folder) / filename
    df.to_csv(path, index=False)
    print(f"Saved → {path.name}  ({df.shape[0]:,} rows × {df.shape[1]} cols)")


# ── Panel summary ──────────────────────────────────────────────────────────────
def panel_summary(df, label, group_col="ric"):
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  Rows       : {len(df):>10,}")
    print(f"  Assets     : {df['ric'].nunique():>10,}")
    print(f"  Date range : {df['date'].min().date()}  →  {df['date'].max().date()}")
    if group_col in df.columns:
        print(f"  Groups     :")
        for grp, cnt in df.groupby(group_col)["ric"].nunique().items():
            print(f"    {grp:<12} {cnt:>6} assets")

## 2. ETF Universe and LSEG Session

The green ETF universe covers renewable energy index funds.
The brown ETF universe covers fossil fuel and commodity energy funds.
All ETFs are listed on NYSE or NASDAQ.

In [8]:
GREEN_ETF_RICS = [
    "ICLN.O", "TAN",    "PBW",    "QCLN.O",
    "FAN",    "PBD",    "ACES.K", "CNRG.K",
    "RNRG.O", "GRID.O", "SMOG.K", "KGRN.K",
]

BROWN_ETF_RICS = [
    "XLE",    "VDE",    "IXC",    "IYE",
    "XOP",    "IEO",    "PXE",    "OIH",
    "XES",    "MLPA.K", "EMLP.N", "ENFR.K",
    "AMLP.K", "USO",    "DBO",    "BNO",
    "UNG",    "FCG.N",
]

ALL_ETF_RICS  = GREEN_ETF_RICS + BROWN_ETF_RICS
ETF_GROUP_MAP = {ric: "green" for ric in GREEN_ETF_RICS}
ETF_GROUP_MAP.update({ric: "brown" for ric in BROWN_ETF_RICS})

print(f"Green ETFs : {len(GREEN_ETF_RICS)}")
print(f"Brown ETFs : {len(BROWN_ETF_RICS)}")
print(f"Total      : {len(ALL_ETF_RICS)}")

Green ETFs : 12
Brown ETFs : 18
Total      : 30


In [9]:
ld.open_session()

# Quick field test to confirm mnemonics are available
_test = lseg_get_history(
    universe=ALL_ETF_RICS[0],
    fields=MARKET_FIELDS,
    start="2024-01-02",
    end="2024-01-10"
)

print("Field test RIC :", ALL_ETF_RICS[0])
print("Rows returned  :", len(_test) if _test is not None else 0)
if _test is not None and not _test.empty:
    print("Columns        :", list(_test.reset_index().columns))

Field test RIC : ICLN.O
Rows returned  : 7
Columns        : ['Date', 'OPEN_PRC', 'HIGH_1', 'LOW_1', 'TRDPRC_1', 'ACVOL_UNS', 'TRNOVR_UNS', 'Company Market Cap']


## 3. ETF Daily Market Data

Daily OHLCV, turnover, and market capitalisation are downloaded for every ETF.
The raw wide frame from LSEG is reshaped to long format, fields are renamed,
types coerced, and return and volatility features are engineered.

**Features constructed:**
- `log_return` — daily log return
- `simple_return` — daily simple return
- `traded_value` — close × volume (liquidity proxy)
- `parkinson_vol` — Parkinson (1980) intraday volatility from high-low range

In [10]:
raw_etf_tables = {}
etf_dl_issues  = []

for ric in ALL_ETF_RICS:
    try:
        raw = lseg_get_history(
            universe=ric,
            fields=MARKET_FIELDS,
            start=START_DATE,
            end=END_DATE
        )
        if raw is None or raw.empty:
            etf_dl_issues.append({"ric": ric, "issue": "empty response"})
        else:
            raw_etf_tables[ric] = raw
    except Exception as err:
        etf_dl_issues.append({"ric": ric, "issue": str(err)})

print(f"Downloaded : {len(raw_etf_tables)} / {len(ALL_ETF_RICS)} ETFs")
if etf_dl_issues:
    print("Issues:")
    print(pd.DataFrame(etf_dl_issues))

Downloaded : 30 / 30 ETFs


In [11]:
etf_panels = []

for ric, raw in raw_etf_tables.items():
    df = wide_to_long(raw)
    if df["ric"].eq("").all() or df["ric"].isna().all():
        df["ric"] = ric.upper()
    df = rename_fields(df)
    df = coerce_types(df)
    df["asset_group"] = ETF_GROUP_MAP.get(ric, "etf")
    etf_panels.append(df)

etf_daily = (
    pd.concat(etf_panels, ignore_index=True)
    .drop_duplicates(subset=["ric","date"])
    .sort_values(["ric","date"])
    .reset_index(drop=True)
)

etf_daily = add_features(etf_daily, group_col="ric")

save_csv(etf_daily, RAW_DIR, "raw_etf_daily.csv")
panel_summary(etf_daily, "ETF daily — raw", group_col="asset_group")

Saved → raw_etf_daily.csv  (42,600 rows × 15 cols)

────────────────────────────────────────────────────────────
  ETF daily — raw
────────────────────────────────────────────────────────────
  Rows       :     42,600
  Assets     :         30
  Date range : 2021-01-04  →  2026-08-28
  Groups     :
    brown            18 assets
    green            12 assets


## 4. ETF NAV — Total Net Assets

Total net assets are reported at monthly frequency
by Refinitiv. The series is downloaded, collapsed to one observation per
month, and merged back onto the daily ETF panel.

In [13]:
nav_tables    = {}
nav_dl_issues = []

for ric in ALL_ETF_RICS:
    try:
        raw = lseg_get_history(
            universe=ric,
            fields=NAV_FIELDS,
            start=START_DATE,
            end=END_DATE
        )
        if raw is None or raw.empty:
            nav_dl_issues.append({"ric": ric, "issue": "empty response"})
        else:
            nav_tables[ric] = raw
    except Exception as err:
        nav_dl_issues.append({"ric": ric, "issue": str(err)})

print(f"NAV downloaded : {len(nav_tables)} / {len(ALL_ETF_RICS)} ETFs")
if nav_dl_issues:
    print(pd.DataFrame(nav_dl_issues))

NAV downloaded : 30 / 30 ETFs


In [14]:
nav_clean_list = []

for ric, raw in nav_tables.items():
    df = raw.copy().reset_index()
    df.columns = [str(c).strip().lower() for c in df.columns]

    if "date" not in df.columns:
        df = df.rename(columns={df.columns[0]: "date"})

    nav_col = [c for c in df.columns if "total net assets" in c or "nav" in c]
    if nav_col:
        df = df.rename(columns={nav_col[0]: "total_net_assets"})
    elif "total_net_assets" not in df.columns:
        df["total_net_assets"] = np.nan

    df["date"]             = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
    df["ric"]              = ric.upper()
    df["month"]            = df["date"].dt.to_period("M").dt.to_timestamp()
    df["total_net_assets"] = pd.to_numeric(df["total_net_assets"], errors="coerce")

    monthly = (
        df[["ric","date","month","total_net_assets"]]
        .dropna(subset=["date"])
        .sort_values(["ric","date"])
        .groupby(["ric","month"], as_index=False)
        .last()
    )
    nav_clean_list.append(monthly)

nav_monthly = pd.concat(nav_clean_list, ignore_index=True)
save_csv(nav_monthly, RAW_DIR, "raw_etf_nav_monthly.csv")

# Merge onto daily ETF panel (match by ric + month)
etf_daily["month"] = etf_daily["date"].dt.to_period("M").dt.to_timestamp()
etf_daily = etf_daily.merge(
    nav_monthly[["ric","month","total_net_assets"]],
    on=["ric","month"],
    how="left"
)

print(f"ETF panel after NAV merge: {etf_daily.shape}")
print(f"NAV coverage: {etf_daily['total_net_assets'].notna().mean()*100:.1f}%")

Saved → raw_etf_nav_monthly.csv  (2,010 rows × 4 cols)
ETF panel after NAV merge: (42600, 17)
NAV coverage: 98.6%


## 5. Commodity Benchmarks — Brent, WTI, and Natural Gas

Daily OHLCV and turnover are downloaded for three commodity benchmarks.

| Asset | LSEG RIC |
|---|---|
| Brent crude | `LCOc1` |
| WTI crude | `CLc1` |
| Natural gas | `NGc1` |

The same Parkinson intraday volatility and log return features are
constructed for each commodity. The panel is then pivoted wide so each
asset becomes its own column set in the final panel —
e.g. `brent_close`, `wti_log_return`, `natural_gas_parkinson_vol`.

In [15]:
COMMODITY_RICS = {
    "brent":       "LCOc1",
    "wti":         "CLc1",
    "natural_gas": "NGc1",
}

raw_commodity_tables = {}
commodity_dl_issues  = []

for asset_name, ric in COMMODITY_RICS.items():
    try:
        raw = lseg_get_history(
            universe=ric,
            fields=COMMODITY_FIELDS,
            start=START_DATE,
            end=END_DATE
        )
        if raw is None or raw.empty:
            commodity_dl_issues.append({"asset": asset_name, "ric": ric, "issue": "empty"})
        else:
            raw_commodity_tables[asset_name] = raw
    except Exception as err:
        commodity_dl_issues.append({"asset": asset_name, "ric": ric, "issue": str(err)})

print(f"Downloaded : {len(raw_commodity_tables)} / {len(COMMODITY_RICS)} commodities")
if commodity_dl_issues:
    print(pd.DataFrame(commodity_dl_issues))

Downloaded : 3 / 3 commodities


In [16]:
commodity_panels = []

for asset_name, raw in raw_commodity_tables.items():
    df = raw.copy().reset_index()
    df.columns = [str(c).strip().lower() for c in df.columns]

    for alias in ["date","index","timestamp"]:
        if alias in df.columns:
            df = df.rename(columns={alias: "date"})
            break

    df = df.rename(columns={
        "open_prc":   "open",
        "high_1":     "high",
        "low_1":      "low",
        "trdprc_1":   "close",
        "acvol_uns":  "volume",
        "trnovr_uns": "turnover",
    })

    for c in ["open","high","low","close","volume","turnover"]:
        if c not in df.columns:
            df[c] = np.nan

    df = df[["date","open","high","low","close","volume","turnover"]].copy()
    df["date"]  = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
    df["asset"] = asset_name
    df["ric"]   = COMMODITY_RICS[asset_name]

    for c in ["open","high","low","close","volume","turnover"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    commodity_panels.append(df)

commodity_daily = (
    pd.concat(commodity_panels, ignore_index=True)
    .drop_duplicates(subset=["asset","date"])
    .sort_values(["asset","date"])
    .reset_index(drop=True)
)

commodity_daily = add_features(commodity_daily, group_col="asset")
save_csv(commodity_daily, RAW_DIR, "raw_commodity_daily.csv")

print(f"\nCommodity panel: {commodity_daily.shape}")
print(commodity_daily.groupby("asset")[["date"]].agg(["min","max"]))

Saved → raw_commodity_daily.csv  (4,305 rows × 14 cols)

Commodity panel: (4305, 14)
                  date           
                   min        max
asset                            
brent       2021-01-04 2026-08-28
natural_gas 2021-01-04 2026-08-28
wti         2021-01-04 2026-08-28


## 6. Firm Universe

Green and brown firm lists are loaded from CSV.
Both files contain pre-screened NYSE and NASDAQ listings —
no exchange filter is applied here.
A `green_dummy` flag (1 = green, 0 = brown) is added for modelling convenience.

In [17]:
green_raw = pd.read_csv(GREEN_FIRM_FILE, encoding="utf-8-sig")
brown_raw = pd.read_csv(BROWN_FIRM_FILE, encoding="utf-8-sig")

print("Green CSV columns:", green_raw.columns.tolist())
print("Brown CSV columns:", brown_raw.columns.tolist())

def prep_firm_list(df, energy_type):
    # Handle both CSV formats (green has 'Identifier' OR 'Ticker')
    if "Identifier" in df.columns:
        id_col = "Identifier"
    elif "Ticker" in df.columns:
        id_col = "Ticker"
    else:
        raise ValueError(f"No identifier column found. Columns: {df.columns.tolist()}")

    name_col = "Company Name" if "Company Name" in df.columns else df.columns[1]

    out = df[[id_col, name_col]].copy()
    out = out.rename(columns={id_col: "ric", name_col: "company_name"})
    out["ric"]          = out["ric"].astype(str).str.strip()
    out["company_name"] = out["company_name"].astype(str).str.strip()
    out["energy_type"]  = energy_type
    out["green_dummy"]  = 1 if energy_type == "green" else 0

    # Drop rows with missing identifiers
    out = out[out["ric"].notna() & (out["ric"] != "") & (out["ric"] != "nan")]
    return out.drop_duplicates(subset=["ric"]).reset_index(drop=True)

green_firms = prep_firm_list(green_raw, "green")
brown_firms = prep_firm_list(brown_raw, "brown")

firm_universe = (
    pd.concat([green_firms, brown_firms], ignore_index=True)
    .replace("", pd.NA)
    .dropna(subset=["ric","company_name","energy_type"])
    .drop_duplicates(subset=["ric"])
    .sort_values(["energy_type","company_name"])
    .reset_index(drop=True)
)

print(f"\nFirm universe: {len(firm_universe)} firms")
print(firm_universe["energy_type"].value_counts())
print()
print(firm_universe.head(10).to_string(index=False))

Green CSV columns: ['Ticker', 'Identifier', 'Company Name', 'Country of Exchange', 'TRBC Industry Name', 'GICS Industry Name']
Brown CSV columns: ['Identifier', 'Company Name', 'Country of Exchange', 'TRBC Industry Name', 'Exchange Name']

Firm universe: 338 firms
energy_type
brown    188
green    150
Name: count, dtype: int64

    ric          company_name energy_type  green_dummy
 APA.OQ         APA Corp (US)       brown            0
ANNA.OQ           AleAnna Inc       brown            0
 AMPY.N   Amplify Energy Corp       brown            0
   AM.N Antero Midstream Corp       brown            0
   AR.N Antero Resources Corp       brown            0
 AROC.N          Archrock Inc       brown            0
  ASC.N Ardmore Shipping Corp       brown            0
ARKO.OQ            Arko Corp.       brown            0
 APC.OQ   Arko Petroleum Corp       brown            0
  BKV.N              BKV Corp       brown            0


## 7. Firm Daily Market Data

Daily OHLCV, turnover, and market capitalisation are downloaded for all firms
and processed through the same pipeline used for ETFs.
Firm metadata (company name, energy type, green dummy) is attached after reshaping.

In [18]:
raw_firm_tables = {}
firm_dl_issues  = []

for ric in firm_universe["ric"].tolist():
    try:
        raw = lseg_get_history(
            universe=ric,
            fields=MARKET_FIELDS,
            start=START_DATE,
            end=END_DATE
        )
        if raw is None or raw.empty:
            firm_dl_issues.append({"ric": ric, "issue": "empty response"})
        else:
            raw_firm_tables[ric] = raw
    except Exception as err:
        firm_dl_issues.append({"ric": ric, "issue": str(err)})

print(f"Universe          : {len(firm_universe)} firms")
print(f"Successfully downloaded : {len(raw_firm_tables)}")
if firm_dl_issues:
    print(f"Failed            : {len(firm_dl_issues)}")
    print(pd.DataFrame(firm_dl_issues))

Universe          : 338 firms
Successfully downloaded : 335
Failed            : 3
      ric                                              issue
0  SUNC.N                                     empty response
1     MWH  Unable to resolve all requested identifiers in...
2  SOLV.O  Unable to resolve all requested identifiers in...


In [19]:
firm_panels = []

for ric, raw in raw_firm_tables.items():
    df = wide_to_long(raw)
    if df["ric"].eq("").all() or df["ric"].isna().all():
        df["ric"] = ric.upper()
    df = rename_fields(df)
    df = coerce_types(df)
    firm_panels.append(df)

firm_daily_raw = (
    pd.concat(firm_panels, ignore_index=True)
    .drop_duplicates(subset=["ric","date"])
    .sort_values(["ric","date"])
    .reset_index(drop=True)
)

# Attach firm metadata
firm_daily_raw = firm_daily_raw.merge(
    firm_universe[["ric","company_name","energy_type","green_dummy"]],
    on="ric",
    how="left"
)

firm_daily_raw = add_features(firm_daily_raw, group_col="ric")

save_csv(firm_daily_raw, RAW_DIR, "raw_firm_daily.csv")
panel_summary(firm_daily_raw, "Firm daily — raw", group_col="energy_type")

Saved → raw_firm_daily.csv  (437,873 rows × 17 cols)

────────────────────────────────────────────────────────────
  Firm daily — raw
────────────────────────────────────────────────────────────
  Rows       :    437,873
  Assets     :        335
  Date range : 2016-09-15  →  2026-08-28
  Groups     :
    brown           185 assets
    green           147 assets


## 8. Firm Shares Outstanding

Refinitiv reports shares outstanding at irregular intervals.
The full history is downloaded, then forward-filled to every trading date
so the daily panel has a complete shares series.
Market capitalisation is carried forward for any days with missing values.

In [20]:
raw_shares_tables = {}
shares_dl_issues  = []

for ric in firm_universe["ric"].tolist():
    try:
        raw = lseg_get_history(
            universe=ric,
            fields=SHARES_FIELDS,
            start=START_DATE,
            end=END_DATE
        )
        if raw is None or raw.empty:
            shares_dl_issues.append({"ric": ric, "issue": "empty response"})
        else:
            raw_shares_tables[ric] = raw
    except Exception as err:
        shares_dl_issues.append({"ric": ric, "issue": str(err)})

print(f"Shares downloaded : {len(raw_shares_tables)} / {len(firm_universe)} firms")
if shares_dl_issues:
    print(f"Issues            : {len(shares_dl_issues)}")
    print(pd.DataFrame(shares_dl_issues).head(15))

Shares downloaded : 333 / 338 firms
Issues            : 5
      ric                                              issue
0  SUNC.N  Unable to resolve all requested identifiers in...
1  BEPH.N                                     empty response
2   ELC.N                                     empty response
3     MWH  Unable to resolve all requested identifiers in...
4  SOLV.O  Unable to resolve all requested identifiers in...


In [21]:
def clean_shares(table_dict):
    records = []
    for ric, raw in table_dict.items():
        df = raw.copy().reset_index()
        df.columns = [str(c).strip().lower() for c in df.columns]

        for alias in ["date","index","timestamp"]:
            if alias in df.columns:
                df = df.rename(columns={alias: "date"})
                break

        shares_col = next(
            (c for c in df.columns if "shares" in c and c != "date"), None
        )
        if shares_col is None:
            continue

        df = df.rename(columns={shares_col: "shares_out"})
        df["ric"]        = ric.upper()
        df["date"]       = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
        df["shares_out"] = pd.to_numeric(df["shares_out"], errors="coerce")

        records.append(df[["ric","date","shares_out"]].dropna(subset=["date"]))

    if not records:
        print("WARNING: no shares data returned.")
        return pd.DataFrame(columns=["ric","date","shares_out"])

    return (
        pd.concat(records, ignore_index=True)
        .drop_duplicates(subset=["ric","date"])
        .sort_values(["ric","date"])
        .reset_index(drop=True)
    )

shares_clean = clean_shares(raw_shares_tables)
save_csv(shares_clean, RAW_DIR, "raw_firm_shares.csv")

# Forward-fill shares onto the daily firm panel
all_dates = pd.DataFrame(
    {"date": pd.date_range(START_DATE, END_DATE, freq="B")}
)
shares_ff = (
    shares_clean
    .groupby("ric")
    .apply(lambda g: (
        g.set_index("date")
         .reindex(all_dates["date"])
         .ffill()
         .reset_index()
         .rename(columns={"index": "date"})
    ))
    .reset_index(drop=True)
)

firm_daily_raw = firm_daily_raw.merge(
    shares_ff[["ric","date","shares_out"]],
    on=["ric","date"],
    how="left"
)

if "company market cap" in firm_daily_raw.columns:
    firm_daily_raw = firm_daily_raw.rename(
        columns={"company market cap": "market_cap"}
    )

# Forward-fill market cap within each firm
firm_daily_raw["market_cap"] = (
    firm_daily_raw.groupby("ric")["market_cap"]
    .transform(lambda s: s.ffill())
)

firm_daily = firm_daily_raw.copy()
save_csv(firm_daily, RAW_DIR, "raw_firm_daily_with_shares.csv")

print(f"\nShares coverage     : {firm_daily['shares_out'].notna().mean()*100:.1f}%")
print(f"Market cap coverage : {firm_daily['market_cap'].notna().mean()*100:.1f}%")

Saved → raw_firm_shares.csv  (432,729 rows × 3 cols)
Saved → raw_firm_daily_with_shares.csv  (437,873 rows × 18 cols)

Shares coverage     : 98.9%
Market cap coverage : 99.7%


In [22]:
print("firm_daily_raw columns:")
print(firm_daily_raw.columns.tolist())

firm_daily_raw columns:
['date', 'open', 'high', 'low', 'close', 'volume', 'turnover', 'market_cap', 'ric', 'company_name', 'energy_type', 'green_dummy', 'log_return', 'simple_return', 'traded_value', 'log_hl_range', 'parkinson_vol', 'shares_out']


## 9. Yahoo Finance Controls — VIX and OVX

Two market uncertainty indices are downloaded from Yahoo Finance:

- **VIX** (`^VIX`) — CBOE equity market volatility index
- **OVX** (`^OVX`) — CBOE crude oil volatility index

The risk-free rate is sourced from FRED via the Fama-French factors
(notebook 04) rather than Yahoo Finance, ensuring consistency with
the factor construction.

In [23]:
def download_yahoo(symbol, col_name, start, end):
    raw = yf.download(symbol, start=start, end=end,
                      auto_adjust=False, progress=False)

    if raw is None or raw.empty:
        return pd.DataFrame(columns=["date", col_name])

    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = ["_".join(str(l) for l in c if l).lower()
                       for c in raw.columns]
    else:
        raw.columns = [str(c).lower() for c in raw.columns]

    raw = raw.reset_index()
    raw = raw.rename(columns={raw.columns[0]: "date"})

    close_col = next((c for c in raw.columns if "close" in c), None)
    if close_col is None:
        raw[col_name] = np.nan
    else:
        raw = raw.rename(columns={close_col: col_name})

    raw["date"]   = pd.to_datetime(raw["date"], errors="coerce").dt.normalize()
    raw[col_name] = pd.to_numeric(raw[col_name], errors="coerce")

    return (raw[["date", col_name]]
            .dropna(subset=["date"])
            .sort_values("date")
            .reset_index(drop=True))


vix_daily = download_yahoo("^VIX", "vix_close", START_DATE, END_DATE)
ovx_daily = download_yahoo("^OVX", "ovx_close", START_DATE, END_DATE)

yahoo_controls = (
    vix_daily
    .merge(ovx_daily, on="date", how="outer")
    .sort_values("date")
    .reset_index(drop=True)
)

save_csv(yahoo_controls, RAW_DIR,   "raw_yahoo_controls.csv")
save_csv(yahoo_controls, CLEAN_DIR, "yahoo_controls.csv")

print(f"\nVIX  : {vix_daily['vix_close'].notna().sum()} obs  "
      f"({vix_daily['date'].min().date()} → {vix_daily['date'].max().date()})")
print(f"OVX  : {ovx_daily['ovx_close'].notna().sum()} obs  "
      f"({ovx_daily['date'].min().date()} → {ovx_daily['date'].max().date()})")

Saved → raw_yahoo_controls.csv  (1,421 rows × 3 cols)
Saved → yahoo_controls.csv  (1,421 rows × 3 cols)

VIX  : 1421 obs  (2021-01-04 → 2026-08-28)
OVX  : 1420 obs  (2021-01-04 → 2026-08-28)


## 10. Fama-French Factors

The complete daily Fama-French factor series — covering January 2021 through
April 2026 including the manually constructed March–April 2026 gap period —
is read from the output of notebook 04.

The gap period factors are constructed following the standard Fama-French (1993)
methodology using LSEG total return data, and are validated against the French
library series over the overlapping period before concatenation.

In [24]:
from pathlib import Path
import pandas as pd
import zipfile
import requests
from io import BytesIO

# ---------------------------------------------------------------------
# Directories
# ---------------------------------------------------------------------

FF5_PATH = CLEAN_DIR / "ff5_daily.csv"

# ---------------------------------------------------------------------
# Download FF5 Daily Factors from Ken French
# ---------------------------------------------------------------------

url = (
    "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
    "F-F_Research_Data_5_Factors_2x3_daily_CSV.zip"
)

r = requests.get(url)
r.raise_for_status()

with zipfile.ZipFile(BytesIO(r.content)) as z:
    txt = z.read(z.namelist()[0]).decode("utf-8")

# ---------------------------------------------------------------------
# Extract the actual data block
# ---------------------------------------------------------------------

lines = txt.splitlines()

start = None
end = None

for i, line in enumerate(lines):
    if line.startswith(",Mkt-RF"):
        start = i
    elif start is not None and line.strip() == "":
        end = i
        break

df = pd.read_csv(
    BytesIO("\n".join(lines[start:end]).encode()),
    index_col=0,
)

# ---------------------------------------------------------------------
# Clean
# ---------------------------------------------------------------------

df.index = pd.to_datetime(df.index.astype(str), format="%Y%m%d")
df.index.name = "date"

df = (
    df.reset_index()
      .rename(columns={"Mkt-RF": "Mkt_RF"})
      .sort_values("date")
      .drop_duplicates("date")
      .reset_index(drop=True)
)

# Convert percentages to decimal returns
factor_cols = ["Mkt_RF", "SMB", "HML", "RMW", "CMA", "RF"]
df[factor_cols] = df[factor_cols] / 100

# Save cleaned version
df.to_csv(FF5_PATH, index=False)

print(f"Saved cleaned FF5 data to {FF5_PATH}")

Saved cleaned FF5 data to /Users/shirindehghannezhad/Documents/Thesis Project/notebooks/../data/clean/ff5_daily.csv


In [25]:
FF5_PATH = CLEAN_DIR / "ff5_daily.csv"

ff5 = pd.read_csv(FF5_PATH, parse_dates=["date"])

ff5 = (
    ff5.dropna(subset=["date"])
       .drop_duplicates(subset=["date"])
       .sort_values("date")
       .reset_index(drop=True)
)

print(f"FF5 factors loaded from : {FF5_PATH.name}")
print(f"Shape      : {ff5.shape}")
print(f"Date range : {ff5['date'].min().date()} → {ff5['date'].max().date()}")
print(f"Columns    : {ff5.columns.tolist()}")

print("\nSample (last 5 rows):")
print(ff5.tail().round(6).to_string(index=False))

FF5 factors loaded from : ff5_daily.csv
Shape      : (15897, 7)
Date range : 1963-07-01 → 2026-08-31
Columns    : ['date', 'Mkt_RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']

Sample (last 5 rows):
      date    Mkt_RF       SMB       HML       RMW       CMA       RF
2026-08-25  0.003100  0.000000 -0.003300 -0.008800 -0.003000 0.000100
2026-08-26 -0.000600 -0.001300  0.002000  0.002600  0.004600 0.000100
2026-08-27  0.007200 -0.001700 -0.011400 -0.007800 -0.008900 0.000100
2026-08-28 -0.003400 -0.003700  0.002800  0.016600  0.004900 0.000100
2026-08-31 -0.003300 -0.002900 -0.003900 -0.010900  0.000300 0.000100


## 11. Final Panels and Save

Yahoo Finance controls, Fama-French factors, and commodity prices are merged
into both the ETF and firm daily panels. Commodity data is pivoted wide so
each asset produces its own column set —
e.g. `brent_close`, `wti_log_return`, `natural_gas_parkinson_vol`.

An `excess_return` column is computed as the log return minus the daily
risk-free rate from the Fama-French library.

In [26]:
COMMODITY_PIVOT_COLS = [
    "close","log_return","simple_return",
    "parkinson_vol","volume","turnover"
]

commodity_wide = commodity_daily.pivot_table(
    index="date",
    columns="asset",
    values=COMMODITY_PIVOT_COLS,
    aggfunc="first"
)

# Flatten: ("close","brent") → "brent_close"
commodity_wide.columns = [
    f"{asset}_{field}" for field, asset in commodity_wide.columns
]
commodity_wide = (
    commodity_wide.reset_index()
    .sort_values("date")
    .reset_index(drop=True)
)

print("Commodity wide columns:")
print(commodity_wide.columns.tolist())

Commodity wide columns:
['date', 'brent_close', 'natural_gas_close', 'wti_close', 'brent_log_return', 'natural_gas_log_return', 'wti_log_return', 'brent_parkinson_vol', 'natural_gas_parkinson_vol', 'wti_parkinson_vol', 'brent_simple_return', 'natural_gas_simple_return', 'wti_simple_return', 'brent_turnover', 'brent_volume', 'natural_gas_volume', 'wti_volume']


In [27]:
controls = (
    yahoo_controls
    .merge(ff5, on="date", how="outer")
    .merge(commodity_wide, on="date", how="outer")
    .sort_values("date")
    .reset_index(drop=True)
)

controls = controls.rename(columns={"Mkt-RF": "Mkt_RF"})

print(f"Controls table : {controls.shape}")
print(f"Date range     : {controls['date'].min().date()} → {controls['date'].max().date()}")
print(f"Columns ({len(controls.columns)}):")
print(controls.columns.tolist())

Controls table : (15939, 25)
Date range     : 1963-07-01 → 2026-08-31
Columns (25):
['date', 'vix_close', 'ovx_close', 'Mkt_RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF', 'brent_close', 'natural_gas_close', 'wti_close', 'brent_log_return', 'natural_gas_log_return', 'wti_log_return', 'brent_parkinson_vol', 'natural_gas_parkinson_vol', 'wti_parkinson_vol', 'brent_simple_return', 'natural_gas_simple_return', 'wti_simple_return', 'brent_turnover', 'brent_volume', 'natural_gas_volume', 'wti_volume']


In [28]:
etf_panel = (
    etf_daily
    .merge(controls, on="date", how="left")
    .sort_values(["ric","date"])
    .reset_index(drop=True)
)

etf_panel["excess_return"] = etf_panel["log_return"] - etf_panel["RF"].fillna(0)

save_csv(etf_panel, CLEAN_DIR, "etf_panel_final.csv")
panel_summary(etf_panel, "ETF panel — final", group_col="asset_group")

Saved → etf_panel_final.csv  (42,600 rows × 42 cols)

────────────────────────────────────────────────────────────
  ETF panel — final
────────────────────────────────────────────────────────────
  Rows       :     42,600
  Assets     :         30
  Date range : 2021-01-04  →  2026-08-28
  Groups     :
    brown            18 assets
    green            12 assets


In [29]:
firm_panel = (
    firm_daily
    .merge(controls, on="date", how="left")
    .sort_values(["ric","date"])
    .reset_index(drop=True)
)

firm_panel["excess_return"] = firm_panel["log_return"] - firm_panel["RF"].fillna(0)

save_csv(firm_panel, CLEAN_DIR, "firm_panel_final.csv")
panel_summary(firm_panel, "Firm panel — final", group_col="energy_type")

Saved → firm_panel_final.csv  (437,873 rows × 43 cols)

────────────────────────────────────────────────────────────
  Firm panel — final
────────────────────────────────────────────────────────────
  Rows       :    437,873
  Assets     :        335
  Date range : 2016-09-15  →  2026-08-28
  Groups     :
    brown           185 assets
    green           147 assets


In [30]:
print("\n" + "="*60)
print("  COLLECTION COMPLETE")
print("="*60)
print(f"\n  Green firms   : {firm_panel[firm_panel['energy_type']=='green']['ric'].nunique()}")
print(f"  Brown firms   : {firm_panel[firm_panel['energy_type']=='brown']['ric'].nunique()}")
print(f"  Green ETFs    : {etf_panel[etf_panel['asset_group']=='green']['ric'].nunique()}")
print(f"  Brown ETFs    : {etf_panel[etf_panel['asset_group']=='brown']['ric'].nunique()}")
print(f"\n  Firm panel    : {firm_panel.shape}")
print(f"  ETF panel     : {etf_panel.shape}")
print(f"  Date range    : {firm_panel['date'].min().date()} → {firm_panel['date'].max().date()}")

ld.close_session()
print("\nLSEG session closed.")


  COLLECTION COMPLETE

  Green firms   : 147
  Brown firms   : 185
  Green ETFs    : 12
  Brown ETFs    : 18

  Firm panel    : (437873, 43)
  ETF panel     : (42600, 42)
  Date range    : 2016-09-15 → 2026-08-28

LSEG session closed.
